# 00 - Download Papers by DOI

Downloads article metadata and full text for all DOIs in the dataset.

## Supported publishers

| Publisher | DOI prefix | What we fetch | API |
|-----------|-----------|---------------|-----|
| **Elsevier** | `10.1016` | Metadata + full text (XML/JSON) | ScienceDirect API (key required) |
| **MDPI** | `10.3390` | Metadata + PDF (open access) | No key |
| **RSC** | `10.1039` | Metadata only | CrossRef |
| **Wiley** | `10.1002` | Metadata only | CrossRef |
| **ACS** | `10.1021` | Metadata only | CrossRef |
| **Any other** | - | Metadata only | CrossRef |

## Setup

1. Set `ELSEVIER_API_KEY` below (or via environment variable).
2. Set `YOUR_EMAIL` - used in CrossRef polite-pool header.
3. Run all cells. Output goes to `downloaded_papers/`.

**Rate limits:** Elsevier = 1 req/s; CrossRef polite pool = ~50 req/s; MDPI = 1 req/s (be polite).

**Note:** Elsevier full-text requires either institutional network access or an API key with
full-text entitlement. If you get 401/403, the API key has metadata-only access.

In [ ]:
import os
import json
import time
import re
import requests
import pandas as pd
from pathlib import Path
from urllib.parse import quote

# ── API key ──────────────────────────────────────────────────────────────────────
# Priority:
#   1. Environment variable  ELSEVIER_API_KEY
#   2. ../API.dat  (line:  API_KEY = "your_key_here")  ← gitignored local file
#   3. Empty string → 401 errors on Elsevier endpoints

def _load_api_key() -> str:
    # 1. env var
    key = os.environ.get('ELSEVIER_API_KEY', '').strip()
    if key:
        return key
    # 2. API.dat next to the data file (one level up from scripts_for_github/)
    api_dat = Path('..') / 'API.dat'
    if api_dat.exists():
        for line in api_dat.read_text(encoding='utf-8').splitlines():
            m = re.search(r'API_KEY\s*=\s*["\']([^"\']+)["\']', line)
            if m:
                return m.group(1).strip()
    return ''

ELSEVIER_API_KEY = _load_api_key()

# Your email for CrossRef polite pool.
# Replace placeholder or set env var CROSSREF_EMAIL.
YOUR_EMAIL = os.environ.get('CROSSREF_EMAIL', 'your@email.com')

# Output directory
OUT_DIR = Path('downloaded_papers')
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / 'elsevier').mkdir(exist_ok=True)
(OUT_DIR / 'mdpi').mkdir(exist_ok=True)
(OUT_DIR / 'metadata').mkdir(exist_ok=True)

DATA_FILE = os.path.join('..', 'go_filtration_data.csv')

# Delays between requests (seconds)
DELAY_ELSEVIER  = 1.0
DELAY_MDPI      = 1.0
DELAY_CROSSREF  = 0.1

if not ELSEVIER_API_KEY:
    print('⚠  ELSEVIER_API_KEY not set.')
    print('   Options:')
    print('     a) Create ../API.dat with line:  API_KEY = "your_key_here"')
    print('     b) PowerShell: $env:ELSEVIER_API_KEY = "your_key_here"  (before starting Jupyter)')
else:
    print(f'API key loaded: {ELSEVIER_API_KEY[:8]}…')
print(f'Output: {OUT_DIR.resolve()}')

In [ ]:
# --- Load DOIs from dataset ---
df = pd.read_csv(DATA_FILE, encoding='utf-8-sig')
dois = sorted(df['doi'].dropna().unique().tolist())
print(f'Total unique DOIs in dataset: {len(dois)}')

# Classify by publisher
def classify_doi(doi):
    doi = doi.strip().lower()
    if doi.startswith('10.1016'):  return 'elsevier'
    if doi.startswith('10.3390'):  return 'mdpi'
    if doi.startswith('10.1039'):  return 'rsc'
    if doi.startswith('10.1002'):  return 'wiley'
    if doi.startswith('10.1021'):  return 'acs'
    if doi.startswith('10.1007'):  return 'springer'
    if doi.startswith('10.1039'):  return 'rsc'
    if doi.startswith('10.1088'):  return 'iop'
    return 'other'

doi_df = pd.DataFrame({'doi': dois})
doi_df['publisher'] = doi_df['doi'].apply(classify_doi)

print('\nDOIs by publisher:')
print(doi_df['publisher'].value_counts().to_string())

## CrossRef - metadata for all DOIs

Free API, no key required. Returns title, authors, journal, year, abstract (when available).

In [ ]:
def crossref_metadata(doi, email=YOUR_EMAIL):
    """Fetch metadata from CrossRef. Returns dict or None on failure."""
    url = f'https://api.crossref.org/works/{quote(doi, safe="")}'
    headers = {'User-Agent': f'GO-ML-research/1.0 (mailto:{email})'}
    try:
        r = requests.get(url, headers=headers, timeout=15)
        if r.status_code == 200:
            data = r.json().get('message', {})
            # Extract key fields
            title = ' '.join(data.get('title', ['']))
            authors = '; '.join(
                f"{a.get('family','')}, {a.get('given','')}"
                for a in data.get('author', [])
            )
            journal = ' '.join(data.get('container-title', ['']))
            year = None
            if 'published-print' in data:
                year = data['published-print'].get('date-parts', [[None]])[0][0]
            elif 'published-online' in data:
                year = data['published-online'].get('date-parts', [[None]])[0][0]
            abstract = data.get('abstract', '')
            # strip HTML tags from abstract
            abstract = re.sub(r'<[^>]+>', '', abstract)
            return {
                'doi': doi,
                'title': title,
                'authors': authors,
                'journal': journal,
                'year': year,
                'abstract': abstract.strip(),
                'type': data.get('type', ''),
                'raw': data,
            }
        else:
            return {'doi': doi, 'error': f'HTTP {r.status_code}'}
    except Exception as e:
        return {'doi': doi, 'error': str(e)}

# Quick test
test = crossref_metadata(dois[0])
print(f"Test DOI: {dois[0]}")
print(f"  Title:   {test.get('title', 'N/A')[:80]}")
print(f"  Journal: {test.get('journal', 'N/A')}")
print(f"  Year:    {test.get('year', 'N/A')}")
print(f"  Authors: {test.get('authors', 'N/A')[:80]}")

In [ ]:
# --- Fetch CrossRef metadata for ALL DOIs ---
meta_cache_file = OUT_DIR / 'metadata' / 'crossref_all.json'

# Load cache if exists
if meta_cache_file.exists():
    with open(meta_cache_file, 'r', encoding='utf-8') as f:
        meta_cache = json.load(f)
    print(f'Loaded cache: {len(meta_cache)} DOIs already downloaded')
else:
    meta_cache = {}

to_fetch = [d for d in dois if d not in meta_cache]
print(f'Downloading {len(to_fetch)} new DOIs from CrossRef...')

for i, doi in enumerate(to_fetch):
    result = crossref_metadata(doi)
    # strip 'raw' from cache to keep file small
    result_save = {k: v for k, v in result.items() if k != 'raw'}
    meta_cache[doi] = result_save
    status = 'OK' if 'error' not in result else f'ERROR: {result["error"]}'
    print(f'  [{i+1}/{len(to_fetch)}] {doi[:40]:<40}  {status}')
    time.sleep(DELAY_CROSSREF)

# Save updated cache
with open(meta_cache_file, 'w', encoding='utf-8') as f:
    json.dump(meta_cache, f, ensure_ascii=False, indent=2)
print(f'\nSaved: {meta_cache_file}')

In [ ]:
# --- Show metadata summary table ---
rows = []
for doi, m in meta_cache.items():
    rows.append({
        'doi':     doi,
        'year':    m.get('year'),
        'journal': m.get('journal', '')[:40],
        'title':   m.get('title', '')[:60],
        'ok':      'error' not in m,
    })
meta_df = pd.DataFrame(rows)
print(f'Successfully downloaded: {meta_df["ok"].sum()} / {len(meta_df)}')
print(f'\nJournals in dataset:')
print(meta_df['journal'].value_counts().head(15).to_string())

## Elsevier - full text (XML) + figures + Supplementary Information

Each article is saved into its own subfolder:
```
fetched_papers/elsevier/
  10_1016_j_memsci_2020_118400/
    article.xml          ← full text XML (view=FULL)
    figures/gr1.jpg, gr2.jpg, ...    ← high-res figures
    si/mmc1.docx, mmc2.pdf, ...      ← supplementary information
    article_aam.pdf                  ← Author Accepted Manuscript (if available)
```

**How it works:**
1. `GET /content/article/doi/{doi}` with `Accept: application/xml`
2. Parse `<object>` elements: `category="high"` → figures, `type="APPLICATION"` → SI
3. Download each via the URL embedded in the `<object>` element text

Requires API key with full-text entitlement (institutional or Elsevier developer account).  
401/403 → key has metadata-only access; 404 → article not on ScienceDirect.

In [ ]:
import xml.etree.ElementTree as ET

ELSEVIER_ARTICLE_URL = 'https://api.elsevier.com/content/article/doi/{doi}'
ELSEVIER_OBJECT_URL  = 'https://api.elsevier.com/content/object/eid/{eid}'


def _download_file(url: str, dest_path, api_key: str, timeout: int = 60) -> tuple:
    """
    Download one binary file from Elsevier object API.
    Returns (ok: bool, size_bytes: int, note: str).
    Mirrors the logic from download_elsevier_xml.py that is known to work.
    """
    headers = {'X-ELS-APIKey': api_key}
    try:
        r = requests.get(url, headers=headers, timeout=(10, timeout))
    except Exception as e:
        return False, 0, str(e)[:80]

    if r.status_code == 200 and r.content:
        # Guard: reject XML/HTML error pages returned as 200
        ct = r.headers.get('Content-Type', '').lower()
        if 'html' in ct or ('xml' in ct and len(r.content) < 2000):
            return False, len(r.content), f'got {ct} ({len(r.content)} B) - likely error page'
        with open(dest_path, 'wb') as f:
            f.write(r.content)
        return True, len(r.content), ''

    return False, 0, f'HTTP {r.status_code}'


def fetch_elsevier_article(doi: str, base_dir, api_key: str = ELSEVIER_API_KEY,
                           verbose: bool = False) -> dict:
    """
    Fetch one Elsevier article into its own subfolder (idempotent):
        {base_dir}/{safe_doi}/article.xml
        {base_dir}/{safe_doi}/figures/gr1.jpg, gr2.jpg, ...
        {base_dir}/{safe_doi}/si/mmc1.docx, mmc2.pdf, ...
    Returns a status dict.
    """
    safe     = doi.replace('/', '_').replace('.', '_')
    art_dir  = Path(base_dir) / safe
    xml_path = art_dir / 'article.xml'
    art_dir.mkdir(exist_ok=True)

    result = {'doi': doi, 'dir': str(art_dir)}

    # 1 - full-text XML (use cached bytes if present)
    if xml_path.exists():
        xml_bytes = xml_path.read_bytes()
    else:
        url   = ELSEVIER_ARTICLE_URL.format(doi=quote(doi, safe=''))
        hdrs  = {'X-ELS-APIKey': api_key, 'Accept': 'application/xml'}
        try:
            resp = requests.get(url, headers=hdrs, timeout=(10, 60))
        except Exception as e:
            result.update(status=-1, error=str(e)); return result

        if resp.status_code in (401, 403):
            result.update(status=resp.status_code, error=resp.text[:200]); return result
        if resp.status_code != 200:
            result.update(status=resp.status_code, error=resp.text[:200]); return result

        xml_bytes = resp.content
        xml_path.write_bytes(xml_bytes)
        time.sleep(DELAY_ELSEVIER)

    result['status']    = 200
    result['xml_chars'] = len(xml_bytes)

    # 2 - parse <object> elements (namespace-agnostic, same as download_elsevier_xml.py)
    try:
        root = ET.fromstring(xml_bytes)
    except ET.ParseError as e:
        result['warning'] = f'XML parse error: {e}'; return result

    objects = [el for el in root.iter()
               if el.tag.endswith('}object') or el.tag == 'object']

    fig_dir = art_dir / 'figures'
    si_dir  = art_dir / 'si'

    fig_count = si_count = aam = 0
    result['download_errors'] = []

    for obj in objects:
        category = obj.get('category', '')
        obj_type = obj.get('type', '')
        ref      = obj.get('ref', '')
        obj_url  = (obj.text or '').strip()
        if not obj_url:
            continue

        if category == 'high' and obj_type == 'IMAGE-HIGH-RES':
            fig_dir.mkdir(exist_ok=True)
            fname = f'{ref}.jpg'
            dest  = fig_dir / fname
            if dest.exists():
                fig_count += 1; continue
            ok, sz, note = _download_file(obj_url, dest, api_key)
            time.sleep(DELAY_ELSEVIER)
            if ok:
                fig_count += 1
                if verbose: print(f'    FIG  {fname}  {sz//1024} KB')
            else:
                result['download_errors'].append(f'{fname}: {note}')
                if verbose: print(f'    FIG  FAIL {fname}: {note}')

        elif obj_type == 'AAM-PDF':
            dest = art_dir / 'article_aam.pdf'
            if not dest.exists():
                ok, sz, _ = _download_file(obj_url, dest, api_key)
                time.sleep(DELAY_ELSEVIER)
                aam = int(ok)

        elif obj_type == 'APPLICATION':
            si_dir.mkdir(exist_ok=True)
            fname = obj_url.split('?')[0].rstrip('/').split('/')[-1] or f'{ref}.docx'
            dest  = si_dir / fname
            if dest.exists():
                si_count += 1; continue
            ok, sz, note = _download_file(obj_url, dest, api_key)
            time.sleep(DELAY_ELSEVIER)
            if ok:
                si_count += 1
                if verbose: print(f'    SI   {fname}  {sz//1024} KB')
            else:
                result['download_errors'].append(f'{fname}: {note}')
                if verbose: print(f'    SI   FAIL {fname}: {note}')

    result['n_figures']    = fig_count
    result['n_si']         = si_count
    result['images_saved'] = fig_count
    result['si_saved']     = si_count
    return result


# --- Quick test: probe first cached XML ---
els_dois = doi_df[doi_df['publisher'] == 'elsevier']['doi'].tolist()
print(f'Elsevier DOIs in dataset: {len(els_dois)}')

if els_dois:
    xml_path_test = (OUT_DIR / 'elsevier'
                     / els_dois[0].replace('/', '_').replace('.', '_')
                     / 'article.xml')
    if xml_path_test.exists():
        root_t  = ET.fromstring(xml_path_test.read_bytes())
        objects = [el for el in root_t.iter()
                   if el.tag.endswith('}object') or el.tag == 'object']
        highs   = [o for o in objects if o.get('category') == 'high' and o.get('type') == 'IMAGE-HIGH-RES']
        suppl   = [o for o in objects if o.get('type') == 'APPLICATION']
        print(f'\nCached XML for {els_dois[0]}:')
        print(f'  High-res figures: {len(highs)}')
        print(f'  SI files        : {len(suppl)}')
        if highs:
            print(f'  Example URL     : {(highs[0].text or "").strip()[:90]}')
    else:
        print('No cached XML yet - run batch cell to fetch.')

In [ ]:
import shutil

# --- Batch fetch all Elsevier articles (XML + figures + SI, one folder per article) ---
# Idempotent: cached article.xml is reused; only missing figure/SI files are downloaded.
# Set FORCE_REFETCH_FIGURES=True to delete and re-download existing figure folders
# (useful if previous runs left empty/corrupt files).
FORCE_REFETCH_FIGURES = True

els_base_dir = OUT_DIR / 'elsevier'
els_results  = []

print(f'Fetching {len(els_dois)} Elsevier articles...')
print('401/403 = no entitlement;  404 = not on ScienceDirect\n')

for i, doi in enumerate(els_dois):
    safe    = doi.replace('/', '_').replace('.', '_')
    art_dir = els_base_dir / safe

    # Wipe corrupt figure/SI dirs from previous failed runs
    if FORCE_REFETCH_FIGURES:
        for sub in ('figures', 'images', 'si'):   # 'images' = old name from earlier attempt
            d = art_dir / sub
            if d.exists():
                shutil.rmtree(d)

    r = fetch_elsevier_article(doi, els_base_dir, verbose=(i == 0))
    els_results.append(r)

    if r.get('status') == 200:
        note = (f"img={r.get('images_saved',0)}/{r.get('n_figures',0)}  "
                f"SI={r.get('si_saved',0)}/{r.get('n_si',0)}")
        errs = r.get('download_errors', [])
        if errs:
            note += f'  ⚠ {len(errs)} err'
            for e in errs[:2]:
                print(f'    {e}')
        print(f'  [{i+1}/{len(els_dois)}] {doi[:45]}  OK  {note}')
    else:
        print(f'  [{i+1}/{len(els_dois)}] {doi[:45]}  HTTP {r.get("status")}  {r.get("error","")[:60]}')

ok_count   = sum(1 for r in els_results if r.get('status') == 200)
total_imgs = sum(r.get('images_saved', 0) for r in els_results)
total_si   = sum(r.get('si_saved',    0) for r in els_results)
print(f'\nDone: {ok_count}/{len(els_dois)} OK  |  {total_imgs} figures  |  {total_si} SI files')
print('Set FORCE_REFETCH_FIGURES=False on next run to skip already-downloaded files.')

## MDPI - PDF via Unpaywall + Europe PMC fallback

MDPI PDFs are open access but their site blocks bots (Cloudflare 403).  
Strategy:
1. **Unpaywall API** (`api.unpaywall.org`) - returns the best open-access PDF URL, free, no key
2. **Europe PMC** - JATS XML + figure ZIP for articles indexed in PubMed Central
3. Direct MDPI URL - last resort (often blocked)

In [ ]:
import io, zipfile
import xml.etree.ElementTree as ET

BROWSER_UA  = ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
               'AppleWebKit/537.36 (KHTML, like Gecko) '
               'Chrome/124.0.0.0 Safari/537.36')
EPMC_SEARCH = 'https://www.ebi.ac.uk/europepmc/webservices/rest/search'
EPMC_XML    = 'https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML'
EPMC_SUPP   = 'https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/supplementaryFiles'


def unpaywall_pdf_url(doi, email=YOUR_EMAIL):
    """
    Query Unpaywall for the best open-access PDF URL.
    Returns url string or None. Free API, no key - just an email.
    """
    try:
        r = requests.get(f'https://api.unpaywall.org/v2/{quote(doi, safe="")}',
                         params={'email': email},
                         headers={'User-Agent': BROWSER_UA}, timeout=15)
        if r.status_code != 200:
            return None
        data = r.json()
        if not data.get('is_oa'):
            return None
        # Try best_oa_location first, then all locations
        for loc in [data.get('best_oa_location')] + data.get('oa_locations', []):
            if loc and loc.get('url_for_pdf'):
                return loc['url_for_pdf']
    except Exception:
        pass
    return None


def download_pdf(url, dest_path):
    """Download a PDF from url. Returns (ok, size_kb, note)."""
    try:
        r = requests.get(url, headers={'User-Agent': BROWSER_UA}, timeout=(10, 60))
        if r.status_code == 200 and r.content[:4] == b'%PDF':
            Path(dest_path).write_bytes(r.content)
            return True, len(r.content) // 1024, ''
        return False, 0, f'HTTP {r.status_code}, starts: {r.content[:20]}'
    except Exception as e:
        return False, 0, str(e)[:80]


def epmc_get_pmcid(doi):
    """Look up PubMed Central ID for a DOI. Returns pmcid string or None."""
    try:
        r = requests.get(EPMC_SEARCH,
                         params={'query': f'DOI:{doi}', 'format': 'xml', 'resulttype': 'core'},
                         headers={'User-Agent': BROWSER_UA}, timeout=(10, 20))
        if r.status_code != 200:
            return None
        root = ET.fromstring(r.content)
        el   = root.find('.//pmcid')
        return el.text.strip() if el is not None and el.text else None
    except Exception:
        return None


def epmc_fetch(doi, out_dir):
    """
    Download JATS XML + figure ZIP from Europe PMC.
    Returns (status: str, notes: str, fig_count: int).
    """
    pmcid = epmc_get_pmcid(doi)
    if not pmcid:
        return 'not_in_pmc', 'no PMCID', 0

    art_dir = Path(out_dir) / doi.replace('/', '_').replace('.', '_')
    art_dir.mkdir(exist_ok=True)

    try:
        r = requests.get(EPMC_XML.format(pmcid=pmcid),
                         headers={'User-Agent': BROWSER_UA}, timeout=(10, 30))
    except Exception as e:
        return 'error', str(e), 0
    if r.status_code != 200:
        return 'error', f'XML HTTP {r.status_code}', 0

    (art_dir / 'article.xml').write_bytes(r.content)

    fig_dir = art_dir / 'figures'
    fig_count = 0
    try:
        sr = requests.get(EPMC_SUPP.format(pmcid=pmcid),
                          headers={'User-Agent': BROWSER_UA},
                          timeout=(10, 30), stream=True)
        if sr.status_code == 200:
            chunks, total = [], 0
            for chunk in sr.iter_content(65536):
                chunks.append(chunk); total += len(chunk)
                if total > 50 * 1024 * 1024:
                    break
            sr.close()
            raw = b''.join(chunks)
            if len(raw) > 100:
                fig_dir.mkdir(exist_ok=True)
                zf = zipfile.ZipFile(io.BytesIO(raw))
                for zname in zf.namelist():
                    ext  = os.path.splitext(zname)[1].lower()
                    base = os.path.basename(zname)
                    if not base: continue
                    if ext in ('.jpg', '.jpeg', '.png'):
                        (fig_dir / base).write_bytes(zf.read(zname))
                        fig_count += 1
    except Exception:
        pass

    return 'ok', f'pmcid={pmcid}, {fig_count} figs', fig_count


# --- Quick test ---
mdpi_dois = doi_df[doi_df['publisher'] == 'mdpi']['doi'].tolist()
print(f'MDPI DOIs in dataset: {len(mdpi_dois)}')

if mdpi_dois:
    doi_t = mdpi_dois[0]
    print(f'\nTesting Unpaywall for: {doi_t}')
    pdf_url = unpaywall_pdf_url(doi_t)
    print(f'  PDF URL: {pdf_url}')
    if pdf_url:
        ok, kb, note = download_pdf(pdf_url, OUT_DIR / 'mdpi' / (doi_t.replace('/','_').replace('.','_') + '.pdf'))
        print(f'  Download: {"OK" if ok else "FAIL"}  {kb} KB  {note}')
    else:
        print('  Not in Unpaywall - trying Europe PMC...')
        st, notes, figs = epmc_fetch(doi_t, OUT_DIR / 'mdpi')
        print(f'  PMC: {st}  {notes}')

In [ ]:
# --- Batch download MDPI articles (Unpaywall → Europe PMC fallback) ---
mdpi_dir     = OUT_DIR / 'mdpi'
mdpi_results = []

print(f'Downloading {len(mdpi_dois)} MDPI articles...')

for i, doi in enumerate(mdpi_dois):
    safe     = doi.replace('/', '_').replace('.', '_')
    pdf_path = mdpi_dir / (safe + '.pdf')
    pmc_dir  = mdpi_dir / safe

    # Already have PDF or PMC XML → skip
    if pdf_path.exists():
        size = pdf_path.stat().st_size
        print(f'  [{i+1}/{len(mdpi_dois)}] {doi[:45]}  CACHED PDF ({size//1024} KB)')
        mdpi_results.append({'doi': doi, 'source': 'unpaywall', 'status': 'ok', 'path': str(pdf_path)})
        continue
    if (pmc_dir / 'article.xml').exists():
        print(f'  [{i+1}/{len(mdpi_dois)}] {doi[:45]}  CACHED PMC XML')
        mdpi_results.append({'doi': doi, 'source': 'pmc', 'status': 'ok', 'dir': str(pmc_dir)})
        continue

    # 1 - Unpaywall: get open-access PDF URL
    pdf_url = unpaywall_pdf_url(doi)
    time.sleep(0.5)
    if pdf_url:
        ok, kb, note = download_pdf(pdf_url, pdf_path)
        time.sleep(DELAY_MDPI)
        if ok:
            print(f'  [{i+1}/{len(mdpi_dois)}] {doi[:45]}  Unpaywall OK  {kb} KB')
            mdpi_results.append({'doi': doi, 'source': 'unpaywall', 'status': 'ok', 'path': str(pdf_path)})
            continue
        print(f'  [{i+1}/{len(mdpi_dois)}] {doi[:45]}  Unpaywall URL found but download failed: {note}')
    else:
        print(f'  [{i+1}/{len(mdpi_dois)}] {doi[:45]}  not in Unpaywall')

    # 2 Europe PMC fallback (JATS XML + figures)
    print(f'    → trying Europe PMC...')
    st, notes, figs = epmc_fetch(doi, mdpi_dir)
    time.sleep(2.0)

    if st == 'ok':
        print(f'    → PMC OK  {notes}')
        mdpi_results.append({'doi': doi, 'source': 'pmc', 'status': 'ok', 'notes': notes})
    else:
        print(f'    → PMC {st}: {notes}')
        mdpi_results.append({'doi': doi, 'source': 'failed', 'status': st, 'notes': notes})

ok_pdf = sum(1 for r in mdpi_results if r.get('source') == 'unpaywall' and r.get('status') == 'ok')
ok_pmc = sum(1 for r in mdpi_results if r.get('source') == 'pmc' and r.get('status') == 'ok')
print(f'\nDone: {ok_pdf} PDFs via Unpaywall  |  {ok_pmc} via Europe PMC  |  {len(mdpi_dois)} total')